# XGBoost fraud model training

Load processed transaction Parquet data, train a class-weighted binary XGBoost model, evaluate it on a held-out test set, and save the trained model. Transaction identifiers and personal identifying fields are excluded from the model features.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import xgboost as xgb
from pyspark.sql import SparkSession, functions as F

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed" / "transactions_parquet").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate data/processed/transactions_parquet from the notebook directory."
    )

processed_data_path = project_root / "data" / "processed" / "transactions_parquet"
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("FraudXGBoostModelTraining")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

numeric_features = [
    "amt",
    "lat",
    "long",
    "city_pop",
    "merch_lat",
    "merch_long",
    "transaction_hour",
    "transaction_day_of_week",
    "is_weekend",
    "amt_log1p",
    "distance_km",
]
categorical_features = ["category", "gender", "state"]
label_column = "is_fraud"
required_columns = ["trans_num", label_column, *numeric_features, *categorical_features]

source_df = spark.read.parquet(str(processed_data_path))
missing_columns = sorted(set(required_columns) - set(source_df.columns))
if missing_columns:
    raise ValueError(f"Processed Parquet data is missing required columns: {missing_columns}")

training_sdf = (
    source_df
    .select(*required_columns)
    .filter(F.col(label_column).isin(0, 1) & F.col("trans_num").isNotNull())
    .dropDuplicates(["trans_num"])
)
training_pdf = training_sdf.toPandas()
if training_pdf.empty:
    raise ValueError("No labeled transactions were found in the processed Parquet data.")

print(f"Loaded {len(training_pdf):,} unique labeled transactions from {processed_data_path}")
print("Label counts (0 = legitimate, 1 = fraud):")
print(training_pdf[label_column].value_counts().sort_index())

c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\dev_env\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Loaded 230,783 unique labeled transactions from c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\data\processed\transactions_parquet
Label counts (0 = legitimate, 1 = fraud):
is_fraud
0    229015
1      1768
Name: count, dtype: int64


In [7]:
training_pdf.shape

(230783, 16)

In [5]:
features = training_pdf.drop(columns=["trans_num", label_column]).copy()
labels = pd.to_numeric(training_pdf[label_column], errors="raise").astype(np.int8)

for column in numeric_features:
    features[column] = pd.to_numeric(features[column], errors="coerce")
    features[column] = features[column].replace([np.inf, -np.inf], np.nan)
    median = features[column].median()
    features[column] = features[column].fillna(0 if pd.isna(median) else median)

for column in categorical_features:
    features[column] = features[column].fillna("Unknown").astype(str)

features = pd.get_dummies(
    features,
    columns=categorical_features,
    dummy_na=False,
    dtype=np.uint8,
).astype(np.float32)

if set(labels.unique()) != {0, 1}:
    raise ValueError("Training requires both legitimate (0) and fraud (1) examples.")

rng = np.random.default_rng(42)
train_indices, validation_indices, test_indices = [], [], []
for label in (0, 1):
    class_indices = np.flatnonzero(labels.to_numpy() == label)
    if len(class_indices) < 3:
        raise ValueError(
            f"At least 3 examples of each label are needed for stratified train/validation/test splits; "
            f"label {label} has {len(class_indices)}."
        )
    rng.shuffle(class_indices)
    test_size = max(1, int(round(len(class_indices) * 0.15)))
    validation_size = max(1, int(round(len(class_indices) * 0.15)))
    if test_size + validation_size >= len(class_indices):
        test_size = validation_size = 1
    test_indices.extend(class_indices[:test_size])
    validation_indices.extend(class_indices[test_size:test_size + validation_size])
    train_indices.extend(class_indices[test_size + validation_size:])

X_train, X_validation, X_test = (
    features.iloc[train_indices],
    features.iloc[validation_indices],
    features.iloc[test_indices],
)
y_train, y_validation, y_test = (
    labels.iloc[train_indices],
    labels.iloc[validation_indices],
    labels.iloc[test_indices],
)

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
feature_names = list(features.columns)
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dvalidation = xgb.DMatrix(X_validation, label=y_validation, feature_names=feature_names)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=feature_names)

training_evaluations = {}
model = xgb.train(
    {
        "objective": "binary:logistic",
        "eval_metric": ["auc", "aucpr"],
        "tree_method": "hist",
        "max_depth": 6,
        "eta": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": negative_count / positive_count,
        "seed": 42,
        "nthread": -1,
    },
    dtrain,
    num_boost_round=500,
    evals=[(dvalidation, "validation")],
    early_stopping_rounds=30,
    verbose_eval=False,
    evals_result=training_evaluations,
    )

print(f"Training rows: {len(X_train):,}; validation rows: {len(X_validation):,}; test rows: {len(X_test):,}")
print(f"Fraud class weight (scale_pos_weight): {negative_count / positive_count:.2f}")
print(f"Best boosting iteration: {model.best_iteration}")
validation_auc = training_evaluations["validation"]["auc"][model.best_iteration]
print(f"Validation ROC-AUC: {validation_auc:.4f}")

Training rows: 161,549; validation rows: 34,617; test rows: 34,617
Fraud class weight (scale_pos_weight): 129.49
Best boosting iteration: 499


In [6]:
fraud_probabilities = model.predict(
    dtest, iteration_range=(0, model.best_iteration + 1)
)
fraud_predictions = (fraud_probabilities >= 0.5).astype(np.int8)
true_labels = y_test.to_numpy()

# Compute ROC-AUC from average ranks so tied prediction scores are handled correctly.
positive_count = int((true_labels == 1).sum())
negative_count = int((true_labels == 0).sum())
all_ranks = pd.Series(fraud_probabilities).rank(method="average").to_numpy()
positive_ranks = all_ranks[true_labels == 1].sum()
roc_auc = (positive_ranks - positive_count * (positive_count + 1) / 2) / (
    positive_count * negative_count
)
gini = 2 * roc_auc - 1

true_positive = int(((true_labels == 1) & (fraud_predictions == 1)).sum())
true_negative = int(((true_labels == 0) & (fraud_predictions == 0)).sum())
false_positive = int(((true_labels == 0) & (fraud_predictions == 1)).sum())
false_negative = int(((true_labels == 1) & (fraud_predictions == 0)).sum())
precision = true_positive / max(true_positive + false_positive, 1)
recall = true_positive / max(true_positive + false_negative, 1)
f1_score = 2 * precision * recall / max(precision + recall, 1e-12)
accuracy = (true_positive + true_negative) / len(true_labels)

print("Test metrics at a 0.5 fraud-probability threshold:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 score:  {f1_score:.4f}")
print(f"  ROC-AUC:   {roc_auc:.4f}")
print(f"  Gini:      {gini:.4f}")
print("Confusion matrix [[true legitimate, false fraud], [missed fraud, detected fraud]]:")
print(np.array([[true_negative, false_positive], [false_negative, true_positive]]))

feature_importance = (
    pd.Series(model.get_score(importance_type="gain"))
    .reindex(feature_names, fill_value=0)
    .sort_values(ascending=False)
    .head(15)
)
print("Top feature importances:")
print(feature_importance)

model_output_path = project_root / "model_development" / "artifacts" / "xgboost_fraud_model.json"
model_output_path.parent.mkdir(parents=True, exist_ok=True)
model.save_model(str(model_output_path))
print(f"Saved trained model to {model_output_path}")

Test metrics at a 0.5 fraud-probability threshold:
  Accuracy:  0.9959
  Precision: 0.6792
  Recall:    0.8868
  F1 score:  0.7692
  ROC-AUC:   0.9939
  Gini:      0.9877
Confusion matrix [[true legitimate, false fraud], [missed fraud, detected fraud]]:
[[34241   111]
 [   30   235]]
Top feature importances:
category_gas_transport    851.548767
amt                       734.721436
amt_log1p                 645.604004
category_grocery_net      412.303650
category_travel           297.330597
category_home             270.968201
category_entertainment    267.051422
category_food_dining      244.841553
transaction_hour          228.183655
category_misc_pos         195.200455
category_shopping_net     176.520660
state_AK                  168.019592
category_shopping_pos     167.589447
category_grocery_pos      127.540161
category_personal_care    107.701813
dtype: float64
Saved trained model to c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\model_developm